# Pendências

Notebook destinado a investigar algumas pendências que ficaram depois da ingestão dos dados. Sendo elas:

- como aparece o `codigo_escola` mascarado
- `item_adaptado` em 2021
- não resposta no questionário

In [ ]:
from pathlib import Path
import duckdb

In [ ]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PQ = ROOT / "data" / "parquet"

base = f"{PQ}/base_unica/**/*.parquet"
res = f"{PQ}/resultados/**/*.parquet"
part = f"{PQ}/participantes/**/*.parquet"
itens = f"{PQ}/itens/**/*.parquet"

con = duckdb.connect()
print(ROOT)

### Pendência 1: como o `codigo_escola` aparece mascarado

**Objetivo**: Descobrir como o INEP escondeu o código das escolas pequenas nos dados de 2024 e 2025.

**Contexto**: A partir de 2024, a tabela RESULTADOS traz o codigo_escola, que identifica a escola de cada participante. Para proteger a privacidade dos alunos, o INEP mascara esse código quando a escola tem menos de 10 participantes. Numa escola com 3 alunos fazendo o ENEM, saber a escola e a nota seria quase o mesmo que saber a nota de cada aluno.

**O problema**: No `ingestao_nulos.csv`, o `codigo_escola` tem **exatamente** o mesmo percentual de nulos que o município da escola:

| Coluna                      | 2024           | 2025       |
|-----------------------------|----------------|------------|
|`nome_municipio_escola`| 66,98% | 62,93%|
|`codigo_escola` | 66,98% | 62,93%|

Se o mascaramento deixasse o código vazio, o `codigo_escola` teria mais nulos que o município, porque as escolas pequenas teriam o município preenchido e o código vazio. Como os percentuais são idênticos, as escolas mascaradas não estão como nulo: o INEP colocou algum outro valor no lugar do código.

Identificar esses valores falsos é importante porque, se não forem tratados adequadamente, ele vira uma "escola" falsa na análise.


In [ ]:
con.sql(f"""
    SELECT ano, codigo_escola, COUNT(*) AS n
    FROM '{res}'
    WHERE codigo_escola IS NOT NULL
    GROUP BY ALL
    ORDER BY n DESC
    LIMIT 10
""").show()

A tabela mostra que não existe um valor substituto para o código das escolas pequenas. Como pode ser visto acima, os códigos mais frequentes são de escolas reais:
    - todos tem 8 dígitos e começam com 31, que é o código do IBGE para Minas Gerais;
    - O maior tem 554 participantes, um número plausível para uma escola estadual grande;
    - Os mesmos códigos aparecem em 2024 e 2025 com contagens parecidas, como `31245488` e `31167436`.

Se existisse um código de mascaramento, ele estaria no topo da tabela (considerando que ele fosse igualmente aplicado para todas as escolas com menos de 10 participantes).

Agora levanto outra **hipótese**: as escolas mascaradas provavelmente aparecem como nulo também no município e nas outras colunas de escola, e não só no código. Assim os percentuais idênticos mostrados na ingestao_nulos.csv também fariam sentido.

In [ ]:
con.sql(f"""
    SELECT ano,
        COUNT(*) FILTER (WHERE codigo_escola IS NULL AND nome_municipio_escola IS NOT NULL) AS codigo_nulo_municipio_ok,
        COUNT(*) FILTER (WHERE codigo_escola IS NOT NULL AND nome_municipio_escola IS NULL) AS codigo_ok_municipio_nulo
    FROM '{res}' GROUP BY ano ORDER BY ano
""").show()

Assim confirmamos que o código e o município da escola estão sempre preenchidos juntos ou vazio juntos em 2024 e 2025. Para confirmar, vamos fazer mais duas análises. Primeiro, vamos investigar se existem escolas com menos de 10 participantes.

In [ ]:
con.sql(f"""
    WITH por_escola AS (
        SELECT ano, codigo_escola, COUNT(*) AS n
        FROM '{res}' WHERE codigo_escola IS NOT NULL
        GROUP BY ALL
    )
    SELECT ano, MIN(n) AS menor_escola,
        COUNT(*) FILTER (WHERE n < 10) AS escolas_com_menos_de_10,
        COUNT(*) AS total_escolas
    FROM por_escola GROUP BY ano ORDER BY ano
""").show()

O resultado confirma que existem sim escolas com menos de 10 participantes nas provas do ENEM de 2024 e 2025. Chegando ao limite de escolas com apenas 1 participante por edição da prova. Se o INEP mascarasse toda escola com menos de 10 participantes, essas escolas não deveriam aparecer no código.

Depois de um longo período de considerações, reformulei a hipótese. Como a D003 estipulou que a tabela RESULTADOS fosse filtrada por prova **ou** escola em MG, ela também contém alunos de escolas de outros estados que fizeram a prova em Minas, e essas escolas provavelmente devem aparecer com poucos participantes, porque só vemos os alunos delas que vieram fazer a prova aqui. Já para as escolas mineiras, a contagem é completa, já que todos os alunos delas entram pelo filtro da escola.

Vamos então, realizar uma nova consulta agora separando as escolas mineiras das escolas de outros estados.

In [ ]:
con.sql(f"""
    WITH por_escola AS (
        SELECT ano, codigo_escola,
            sigla_estado_escola = 'MG' AS escola_mg,
            COUNT(*) AS n
        FROM '{res}' WHERE codigo_escola IS NOT NULL
        GROUP BY ALL
    )
    SELECT ano, escola_mg,
        MIN(n) AS menor_escola,
        COUNT(*) FILTER (WHERE n < 10) AS escolas_com_menos_de_10,
        COUNT(*) AS total_escolas
    FROM por_escola
    GROUP BY ALL ORDER BY ano, escola_mg
""").show()

Uma parte da última hipótese foi sanada. Para as escolas de fora do estado (`escola_mg = false`), conseguimos ver que mais de 90% delas apresenta menos de 10 alunos fazendo o ENEM no território de Minas Gerais. 

| Ano | Escolas de outros estados | Com menos de 10 |
|------|--------------------------|-----------------|
|2024 | 263 | 242 (92%)|
|2025 | 274 | 251 (92%) |

Isso gera uma nova decisão para a próxima estapa deste projeto: **as análises por escola devem usar apenas as escolas mineiras (`sigla_estado_escola = 'MG'`)**, porque as de fora aparecem incompletas.

Por outro lado, a tabela acima ainda mostra que existem escolas mineiras pequenas que não foram mascaradas.

| Ano | Escolas de MG | Com menos de 10 |
|------|--------------------------|-----------------|
|2024 | 3.326 | 565 (17%)|
|2025 | 3.497 | 489 (14%) |

Então a regra registrada na D002 ("mascarado se a escola tem menos de 10 participantes") não aparece nos dados da forma como foi escrita.

Resolvi reabrir o dicionário de dados de 2024 para verificar se havia alguma observação que eu deixei passar e sim, havia uma observação. O texto que informava sobre a máscara no dicionário diz exatamente o seguinte:

> Informação obtida da base de matrículas do Censo Escolar 2024, a partir do CPF do participante e filtrando as etapas de ensino de prováveis concluintes do ensino médio (27,28,29,37,38,71,32,33,34). O código real será substituído por uma máscara, caso escola tenha menos de 10 participantes na edição do exame.

Como já investigamos a possibilidade uma máscara única ser usada para as escolas com menos de 10 participantes e não encontramos tal máscara, o texto do INEP sugere que as máscaras utilizadas são únicas e que continuam parecendo códigos de escolas reais. Por isso a busca pelo valor mais frequente feita no início dessa investigação não achou nada. Vamos verificar se, de fato, as máscaras seguem o mesmo padrão dos códigos reais das escolas com mais participantes.

In [ ]:
con.sql(f"""
    WITH por_escola AS (
        SELECT ano, codigo_escola, COUNT(*) AS n
        FROM '{res}'
        WHERE codigo_escola IS NOT NULL AND sigla_estado_escola = 'MG'
        GROUP BY ALL
    )
    SELECT ano,
        n < 10 AS menos_de_10,
        length(codigo_escola) AS tamanho,
        codigo_escola LIKE '31%' AS comeca_com_31,
        COUNT(*) AS escolas,
        MIN(codigo_escola) AS exemplo_min,
        MAX(codigo_escola) AS exemplo_max
    FROM por_escola
    GROUP BY ALL ORDER BY ano, menos_de_10, tamanho
""").show()

Essa última análise resolve a pendência 1. O resultado mostra que as escolas com menos de 10 participantes têm o código substituído por uma máscara no formato `600xxxxx`. Os códigos reais, como falado anteriormente, começam com o código da UF no IBGE (31 para Minas Gerais).

**Conclusões**
- A máscara é identificável pelo formato, sem precisar contar o número de participantes.
- Cada escola mascarada recebe uma máscara própria.
- As demais colunas da escola são reais: município, rede e localização continuam valendo para esses alunos.

## Pendência 2 - excesso de nulos em `item_adaptado` de 2021

No `ingestao_nulos.csv`, a coluna `item_adaptado` tem 76% de nulos só em 2021, enquanto nos outros anos tem 0%. Essa coluna indica se a questão foi adaptada, como nas provas para pessoas com deficiência visual, em que itens com imagens às vezes são modificados.

O dicionário de dados de 2021 não trás nenhuma observação sobre isso, então vou fazer algumas análises para tentar entender o que o nulo significa ali. Pode ser que, em 2021, o INEP só preencheu as colunas para os cadernos adaptados e deixou vazio nos regulares. 

Vamos começar olhando nos outros anos para ver como essa coluna é codificada normalmente:

In [ ]:
con.sql(f"""
    SELECT ano, item_adaptado, COUNT(*) AS itens
    FROM '{itens}'
    GROUP BY ALL ORDER BY ano, item_adaptado
""").show(max_rows=50)

Em acordo com o `ingestao_nulos.csv`, apenas em 2021 temos a presença de `NULL` ou `1`. Isso indica que o INEP deixou vazio o campo que nos outros anos era preenchido com `0`.

| Ano | `0` (não adaptado) | `1` (adaptado)   | `NULL` |
|-----|--------------------|------------------|---------|
|2020 | 720 | 315  |   —   |
|2021 | —  |  315  | 990  |
|2022 | 1170  | 135  |  —  |

Como cada caderno de prova tem 45 questões, podemos verificar que os números que aparecem na tabela acima são múltiplos de 45. O que nos mostra que cadernos inteiros são ou não são adaptados:

- *315* adaptados = *7* cadernos. As quantidades de 2020 e 2021 são iguais aqui.
- *990* nulos = *22* cadernos, que seriam os regulares.

**Conclusões**

- Em 2021, o nulo em `item_adaptado` significa "não adaptado".
- Na limpeza de dados, é seguro converter o nulo em `0` só em 2021.
- Uma observação é necessária no `harmonizacao_itens.csv`.

## Pendência 3 - não resposta no questionário socioeconômico

No `ingestao_nulos.csv`, o questionário socioeconômico tem 1,5% de nulos em 2020 e 0% de 2021 em diante. Consigo pensar em duas hipóteses:

- o questionário passou a ser obrigatório a partir de 2021, e todos responderam;
- a não resposta mudou de forma: em vez de ficar vazia, passou a ser uma categoria, como uma letra específica para "não respondeu".

Além disso, em todos os anos, as perguntas sobre os pais costuma ter uma opção "Não sei". Essa opção de resposta cobre uma realidade brasileira da falta de registro ou convivio paterno. Provavelmente a opção aparece mais frequentemente nas perguntas sobre o pai do que nas perguntas sobre a mãe. De qualquer forma, ela não representa um nível de escoladirade, por exemplo, e não pode entrar numa escala ordinal como se fosse. Assim, preciso encontrar uma forma de lidar com ela.

Vamos checar a categoria de cada variável ano a ano:

In [ ]:
cols = ["escolaridade_pai", "escolaridade_mae", 
        "grupo_ocupacao_pai", "grupo_ocupacao_mae", "qtd_moradores", "faixa_renda"]

con.sql(f"""
    UNPIVOT (SELECT ano, {', '.join(cols)} FROM '{base}')
    ON {', '.join(cols)}
    INTO NAME variavel VALUE categoria
""").create_view("quest")

tabela = (con.sql("""
    SELECT variavel, categoria, ano, COUNT(*) AS n
    FROM quest GROUP BY ALL
""").df().pivot_table(index=["variavel", "categoria" ], columns="ano", values="n").fillna(0).astype(int))

tabela

A consulta mostra que não houve acréscimo de categoria em anos posteriores a 2020. Também podemos ver um grande volume de "Não sei" (opção `H`) nos dados referentes à escolaridade e ocupação do pai. O que corrobora a hipótese sobre a falta de convívio paterno. Entretanto, essa análise não foi suficiente para identificar e entender os dados nulos.

Como nosso grupo de interesse são os estudantes concluíntes do ensino médio no ano de realização do ENEM, vamos verificar se os nulos aparecem nos treineiros ou nos concluíntes:

In [ ]:
con.sql(f"""
    SELECT ano,
        faixa_renda IS NULL AS renda_nula,
        treineiro,
        COUNT(*) AS n
    FROM '{base}'
    WHERE ano IN (2020, 2021)
    GROUP BY ALL ORDER BY ano, renda_nula, treineiro
""").show()

São 8.701 participantes em 2020 com `renda_nula = true`, mas 8.700 deles não são treineiros. Portanto, não dá pra resolver esse problema somente excluindo os treineiros da análise. Os nulos aparecem justamente na população de interesse. Vamos ver quantos são concluíntes do ano:

In [ ]:
con.sql(f"""
    SELECT situacao_conclusao, COUNT(*) AS n
    FROM '{base}'
    WHERE ano = 2020 AND faixa_renda IS NULL
    GROUP BY ALL ORDER BY situacao_conclusao
""").show()

A maioria dos nulos de 2020 está fora da população da análise individual. Pelo dicionário do ENEM, os códigos de `situacao_conclusao` são:

| Código | Situação | Nulos em 2020 |
|--------|-----------|--------------|
|1| Já concluíu o ensino médio | 7.150|
|2| **conclui no ano da prova** | 1.548|
|3| conclui depois do ano da prova | 2|
|4| não conluiu e não está cursando | 1|

Como a análise individual que vamos fazer usa apenas os concluíntes do ano (D002), isso significa que dos 8.701 nulos, só 1.548 importam. Vamos entender o peso desses nulos de interesse:

In [ ]:
con.sql(f"""
    SELECT COUNT(*) AS concluintes,
        COUNT(*) FILTER (WHERE faixa_renda IS NULL) AS sem_questionario,
           round(100.0 * COUNT(*) FILTER (WHERE faixa_renda IS NULL) / COUNT(*), 2) AS pct
    FROM '{base}'
    WHERE ano = 2020 AND situacao_conclusao = '2' AND treineiro = '0'
""").show()

São só 1.548 de 125.998 concluíntes, ou 1,23%. É uma fração pequena, num ano só, o que me leva a crer que excluir esses participantes da população de interesse não trará nenhum problema. Entretanto, surge a ideia de que, para não introduzir nenhum tipo de viés, seja interessante ver como as notas desses alunos sem os dados do questionário socioeconômico se comparam com as notas dos demais alunos. Se as notas seguirem uma certa similaridade, podemos descartá-los com mais tranquilidade. Caso contrário, isso pode implicar numa limitação de análise.

In [ ]:
con.sql(f"""
    SELECT faixa_renda IS NULL AS sem_questionario,
        COUNT(nota_cn) AS presentes_cn,
        round(AVG(nota_cn), 1) AS media_cn,
        round(100.0 * COUNT(nota_cn) / COUNT(*), 1) AS pct_presentes
    FROM '{base}'
    WHERE ano = 2020 AND situacao_conclusao = '2' AND treineiro = '0'
    GROUP BY ALL
""").show()

Conseguimos ver que há uma diferença, mas que ainda sim o impacto da exclusão é muito pequeno, talvez insignificante.

**Os participantes sem questionário são diferentes**
|       | Com questionário | Sem questionário |
|-------|----------------|-----------------|
|Média em CN | 505,2 | 492,7 (12,5 pontos abaixo)|
|Presença no 2º dia | 61,9%  | 42,5% |

Quem deixou o questionário em branco faltou mais à prova e teve nota um pouco menor. Faz sentido se pensarmos que talvez seja um grupo menos engajado no exame.

São só 658 notas contra 77.064. Incluindo ou excluindo esse grupo, a média geral de 2020 muda de 505,2 para 505,1. Para a análise de notas, a exclusão parece ser inofensiva.

**Conclusão**

- As respostas "Não sei" nos dados socioeconômicos podem representar um recorte importante da realidade brasileira (ou mineira) e por isso não deve ser ignorada. 
- As não respostas ao questionário vêm de um grupo muito pequeno dentre a população de interesse da análise que vou fazer. Excluí-los não afetará significativamente os resultados que serão obtidos.

